# Octospawn

Autorzy: **Kalina Raczka, Emilia Sarna**

## Opis gry

### Wersja deterministyczna
Gra **Octospawn** jest wariantem Hexapawna rozgrywanym na planszy 4x4. Każdy z graczy dysponuje 4 pionkami na swojej linii startowej. Celem gry jest dojście pionkiem do przeciwległej krawędzi planszy, zbicie wszystkich pionków przeciwnika lub zablokowanie mu możliwości wykonania ruchu. Pionki poruszają się o jedno pole do przodu lub biją po skosie.

### Wersja probabilistyczna
W wariancie probabilistycznym, po każdym ruchu gracza istnieje **10% szansy** na odrodzenie jednego z uprzednio zbitych pionków tego gracza. Odrodzony pionek wraca na swoje oryginalne pole startowe.


In [ ]:
if (random.random() < self.chance or force_resurrect) and self.player.lost_pawns:
    pawn = random.choice(self.player.lost_pawns)
    all_positions = [p[1] for p in self.player.pawns + self.opponent.pawns]

    # Jeśli pole startowe jest puste, odradzamy pionka
    if pawn[0] not in all_positions:
        self.player.lost_pawns.remove(pawn)
        self.player.pawns.append((pawn[0], pawn[0]))
        move_data["resurrected"] = pawn

Implementacja węzła losowego w wariancie probabilistycznym. Kod najpierw sprawdza spełnienie 10% szansy na wskrzeszenie, a następnie weryfikuje, czy oryginalne pole startowe zbitego pionka nie jest zablokowane przez inne figury

### Implementacja zasad gry

In [ ]:
# Funkcja oceniająca stan gry
def scoring(game):
    return -100 if game.lose() else 0

# Metoda warunkująca koniec gry
def lose(self):
    opp = [p[1] for p in self.opponent.pawns]
    # Przegrana następuje gdy przeciwnik doszedł do naszej linii startowej
    # lub gdy aktualny gracz nie ma żadnych legalnych ruchów.
    return any([i == self.opponent.goal_line for i, j in opp]) or (
        self.possible_moves() == []
    )

Algorytm iteruje po możliwych stanach i premiuje te, które unikają spełnienia warunków z funkcji lose() (dojście przeciwnika do celu lub blokada ruchów).

## Zastosowane algorytmy
Porownano 3 warianty:
* Negamax
* Negamax z odcieciem alfa-beta
* ExpectiMiniMax z odcieciem alfa-beta (tylko dla prob=0.1)

Dla algorytmu `ExpectiMiniMax` zmodyfikowano standardowe podejście drzewa przeszukiwań, dodając tzw. węzły szansy (chance nodes), które uwzględniają 10% prawdopodobieństwa na odrodzenie się pionka.

### Negamax
Jest to wariant algorytmu Minimax, który opiera się on na założeniu, że interesy obu graczy są przeciwne (przeciwnik minimalizuje możliwie cel swojego rywala). Natomiast, zamiast pisać osobne funkcje dla gracza maksymalizującego i minimalizującego, Negamax używa jednej funkcji, która zawsze maksymalizuje wynik z perspektywy gracza, którego aktualnie jest tura. Przy każdym ruchu algorytm wybiera ten ruch, którego znaleziony wynik jest największy.

In [ ]:
# mynegamax.py
if bestValue < move_alpha:
    bestValue = move_alpha
    best_move = move
    if depth == origDepth:
        state.ai_move = move


### Negamax z odcieciem alfa-beta
Stanowi optymalizację podstawowego algorytmu Negamax. Wprowadza dwa parametry graniczne: `alfa` (minimalny wynik gwarantowany dla gracza maksymalizującego) oraz `beta` (maksymalny wynik gwarantowany dla gracza minimalizującego). Jeśli w trakcie przeszukiwania drzewa algorytm zorientuje się, że aktualnie rozpatrywana gałąź prowadzi do wyniku gorszego niż już znana bezpieczna alternatywa, natychmiast przerywa dalsze sprawdzanie tej gałęzi (następuje tzw. odcięcie). Zmniejsza to liczbę przeszukiwanych węzłów, redukując czas wyboru ruchu.

In [ ]:
if alpha < move_alpha and pruning:
    alpha = move_alpha
    if alpha >= beta:
        break

### ExpectiMiniMax
To rozszerzenie algorytmu Minimax stworzone specjalnie z myślą o grach zawierających element losowy (np. losowe odrodzenie pionka w Octospawn). Pomiędzy węzłami decyzyjnymi graczy wprowadza "węzły szansy" (chance nodes). Węzeł taki nie wybiera najlepszego ruchu, lecz oblicza wartość oczekiwaną stanu gry, mnożąc wartości możliwych wyników przez ich prawdopodobieństwo wystąpienia (w naszym przypadku 90% na zwykły ruch i 10% na odrodzenie)
W eksperymentach użyto tylko wersji z odcięciem alfa-beta.

In [ ]:
# expectiminimax.py
if moving_player_has_lost_pawns:
    # Symulacja ruchu z wymuszonym odrodzeniem pionka
    game.make_move(move, force_resurrect=True)
    game.switch_player()

    move_alpha_yes = -expectiminimax(game, depth - 1, origDepth, scoring, -beta, -alpha, pruning=pruning)

    if unmake_move:
        game.switch_player()
        game.unmake_move(move)

    # Wartość oczekiwana to średnia ważona prawdopodobieństwem (90% na brak odrodzenia, 10% na odrodzenie)
    move_alpha = (1 - game.chance) * move_alpha_no + (game.chance) * move_alpha_yes
else:
    move_alpha = move_alpha_no

## Eksperymenty i wyniki

Przeprowadziłyśmy serię eksperymentów, w których algorytmy rozgrywały ze sobą po **100 partii**. W każdej parze gier (po połowie) algorytmy zamieniały się stroną rozpoczynającą, co pozwoliło zniwelować przewagę pierwszego gracza przy ocenie.

### Wyniki rozgrywek
Poniższa tabela prezentuje wyniki rozgrywek poszczególnych algorytmów samych ze sobą przy różnych parametrach.

| Algorytm                              | Wygrane Zaczynający | Wygrane Drugi |
|:--------------------------------------|:-------------------:|:-------------:|
| Negamax, d=3, alfa-beta, prob=0.0     |       **100**       |       0       |
| Negamax, d=3, bez alfa-beta, prob=0.0 |       **100**       |       0       |
| Negamax, d=4, alfa-beta, prob=0.0     |       **100**       |       0       |
| Negamax, d=4, bez alfa-beta, prob=0.0 |       **100**       |       0       |
| Negamax, d=3, alfa-beta, prob=0.1     |         73          |      27       |
| Negamax, d=3, bez alfa-beta, prob=0.1 |         78          |      22       |
| Negamax, d=4, alfa-beta, prob=0.1     |         80          |      20       |
| Negamax, d=4, bez alfa-beta, prob=0.1 |         74          |      26       |
| ExpectiMiniMax, d=3, prob=0.1         |         71          |      29       |
| ExpectiMiniMax, d=4, prob=0.1         |         83          |      17       |

Zestawienie wyników wykazuje pełną rozwiązalność gry Octospawn w środowisku deterministycznym (100% wygranych gracza rozpoczynającego). Element losowy (prob = 0.1) zauważalnie redukuje tę przewagę, dając drugiemu graczowi średnio 20-30% szans na wygraną.

### Bezpośrednie starcia algorytmów (prawdopodobieństwo=0.1)

| P1 vs P2                               | Wygrane P1 | Wygrane P2 | Wygrane Zaczynający | Wygrane Drugi  |
|:---------------------------------------|:----------:|:----------:|:-------------------:|:--------------:|
| Negamax(d=3,ab) vs Negamax(d=4,ab)     |     48     |     52     |         80          |       20       |
| Negamax(d=3,ab) vs ExpectiMiniMax(d=3) |     47     |     53     |         75          |       25       |
| Negamax(d=4,ab) vs ExpectiMiniMax(d=4) |     51     |     49     |         73          |       27       |

Zestawienie head-to-head dowodzi, że w praktycznych starciach na głębokości rzędu 3-4, skomplikowany ExpectiMiniMax gra na bardzo zbliżonym poziomie skuteczności do standardowego algorytmu Negamax.

### Średni czas decyzji (w sekundach)

| Algorytm                    | Deterministyczna [s] | Probabilistyczna [s] |
|:----------------------------|:--------------------:|:--------------------:|
| Negamax, d=3, alfa-beta     |        0.0017        |        0.0019        |
| Negamax, d=3, bez alfa-beta |        0.0043        |        0.0043        |
| Negamax, d=4, alfa-beta     |        0.0033        |        0.0035        |
| Negamax, d=4, bez alfa-beta |        0.0158        |        0.0167        |
| ExpectiMiniMax, d=3         |          -           |        0.0038        |
| ExpectiMiniMax, d=4         |          -           |        0.0125        |

## Dyskusja i wnioski

1. **Miażdżąca przewaga pierwszego gracza:**
   W tabeli 1 wyraźnie widać, że w środowisku deterministycznym (`prob=0.0`), bez względu na zastosowany algorytm (Negamax, odcięcie AB, głębokość), **gracz zaczynający wygrywa w 100% przypadków**. Oznacza to, że Octospawn na planszy 4x4 jest grą rozwiązaną (posiada strategię wygrywającą dla gracza rozpoczynającego), a głębokość rzędu 3-4 wystarcza, aby algorytm to zauważył.

2. **Wpływ losowości na szanse drugiego gracza:**
   Wprowadzenie mechaniki probabilistycznej (10% na odrodzenie pionka) zaburza tę idealną strategię. W tym wariancie Drugi gracz zaczyna wygrywać od ok. 17% do 29% rozdań. Zjawisko to wynika z powrotów zbitych pionków do gry, potrafiących całkowicie obrócić sytuację na planszy i zniwelować "matematyczną" przewagę pierwszego ruchu.

3. **Wydajność odcięcia Alfa-Beta (Alpha-Beta Pruning):**
   Analiza czasów decyzji z Tabeli 3 pokazuje przewagę cięcia alfa-beta. Dla głębokości rzędu 4, klasyczny Negamax potrzebuje około `0.016 s` na decyzję, z kolei wariant z odcięciem alfa-beta redukuje ten czas blisko pięciokrotnie (do ok. `0.003 s`). Optymalizacja ta zachowuje identyczną jakość ruchów, oszczędzając czas dzięki nieprzeszukiwaniu suboptymalnych gałęzi.

4. **ExpectiMiniMax kontra Negamax:**
   Zgodnie z naszymi wynikami (Tabela 2), ExpectiMiniMax, mimo poprawnego modelowania prawdopodobieństwa zdarzeń z pomocą węzłów szansy, gra bardzo porównywalnie do prostego algorytmu Negamax (w bezpośrednim starciu na d=4 było to 49 do 51 wygranych). Co więcej, z uwagi na konieczność kalkulowania rozgałęzień (rozpatrywanie scenariusza po odrodzeniu i bez odrodzenia), ExpectiMiniMax jest **kilkukrotnie wolniejszy** w wariancie prob=0.1 na głębokości 4 (`0.0125 s` vs `0.0035 s` dla Negamax AB).
   Wynika to prawdopodobnie z faktu, że 10% szansa na zmartwychwstanie w grze o stosunkowo krótkim czasie trwania rzadko zmienia wycenę samej gałęzi drzewa na tyle mocno, by uzasadnić duże koszty obliczeniowe. W większości partii to Negamax sprawdza się wystarczająco dobrze i znacznie szybciej.

## Napotkane problemy i ich rozwiązania
* **Ograniczenia klasycznego algorytmu Negamax**: Standardowy Negamax z założenia operuje na środowisku w pełni deterministycznym, przez co kompletnie nie uwzględnia zdarzeń losowych. Dopiero wdrożenie algorytmu ExpectiMiniMax pozwoliło nam na pełne i poprawne matematycznie zamodelowanie losowości.

* **Drastyczny spadek wydajności przy większych głębokościach**: Wymóg liczenia wartości oczekiwanej sprawił, że ExpectiMiniMax okazał się algorytmem niezwykle zasobożernym – znacznie bardziej niż Negamax. Lecz przy większych głębokościach (np.7,8), czas oczekiwania na pojedynczy ruch był dość długi. Uniemożliwiło to nam przeprowadzenie dokładniejszych testów.

* **Rozróżnienie perspektywy po ruchu z odrodzeniem:** Logika odradzania pionka wymagała poprawnej identyfikacji tego, *kto stracił pionka*, aby wskrzesić pionka właściwego gracza po wywołaniu `game.switch_player()`.